# 第 1 天 · 环境搭建

> 📓 **这是第 1 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/1)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
%pip install -q httpx python-dotenv
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
try:  # 同步 httpx / openai 在浏览器里走同步 XHR（和课程网页的运行器一致）
    import httpx._transports.jsfetch as _jf
    _jf.can_run_sync = lambda: False
except Exception:
    pass
print("✓ 环境就绪，可以往下运行了")

In [ ]:
# 🔑 这一课要调用大模型：把 Key 填在引号里再运行这一格（只存在这次打开的浏览器内存里，别分享这个 Notebook）
import os
LLM_API_KEY = ""
os.environ["LLM_BASE_URL"] = "https://api.deepseek.com"   # 通义：https://dashscope.aliyuncs.com/compatible-mode/v1
os.environ["LLM_MODEL"] = "deepseek-chat"                 # 通义：qwen-plus
if LLM_API_KEY:
    os.environ["LLM_API_KEY"] = LLM_API_KEY
print("Key：", "已设置" if os.getenv("LLM_API_KEY") else "未设置（会走缺 Key 的报错分支）")

In [ ]:
def main():
    print("Hello from agent-lab!")


if __name__ == "__main__":   # ≈ public static void main，只有直接运行这个文件才执行
    main()

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # 读取 .env，把里面的键值放进环境变量
api_key = os.getenv("LLM_API_KEY")
if api_key:
    print("读到了：", api_key[:6] + "...")  # 只打印前几位，确认读到了
else:
    print("没读到 LLM_API_KEY（本地：检查 .env；App 里：去 ⚙️ 设置 填写）")

In [ ]:
import httpx

resp = httpx.get("https://api.github.com/users/torvalds")
print(resp.status_code)        # 200
data = resp.json()             # 把返回的 JSON 转成 dict
print(data["public_repos"])

In [ ]:
import httpx

resp = httpx.get(
    "https://api.github.com/search/repositories",
    params={"q": "agent", "per_page": 3},     # 会拼成 ?q=agent&per_page=3
    headers={"Accept": "application/vnd.github+json"},
    timeout=10.0,
)
resp.raise_for_status()   # 4xx/5xx 时直接抛异常，≈ RestTemplate 默认行为
for repo in resp.json()["items"]:
    print(repo["full_name"])

In [ ]:
import httpx

resp = httpx.post("https://httpbin.org/post", json={"name": "jiawei"})
print(resp.json()["json"])

In [ ]:
import httpx

with httpx.Client(base_url="https://api.github.com", timeout=10) as client:
    r = client.get("/users/torvalds")
    print(r.json()["name"])

In [ ]:
import httpx

USERNAME = "torvalds"  # TODO: 改成你的 GitHub 用户名


def main() -> None:
    resp = httpx.get(f"https://api.github.com/users/{USERNAME}", timeout=10)
    resp.raise_for_status()
    data = resp.json()
    print(f"{data['login']} 有 {data['public_repos']} 个公开仓库")


if __name__ == "__main__":
    main()